# Experiment C: Error Decomposition (§5.3)

Decomposes classification errors into cross-element and intra-element
categories. Demonstrates that BIM-BPC eliminates 100% of cross-element
errors while intra-element errors remain unchanged.

## Setup

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('../data/experiment_H_results.json', 'r') as f:
    decomp = json.load(f)

print(f"Test images: {decomp['n_test']}")
print(f"Corrected samples: {decomp['corrected_count']}")
print(f"Still wrong after BPC: {decomp['still_wrong_count']}")
print(f"Confident correct: {decomp['confident_correct_count']}")

## Error Decomposition: Cross vs Intra-element

In [ ]:
ed = decomp['error_decomposition']
print('Error Decomposition:')
print('=' * 50)
print(f"{'':20s} {'Baseline':>10s} {'BIM-BPC':>10s}")
print(f"{'Total errors':20s} {ed['baseline']['total']:>10d} {ed['bim_corrected']['total']:>10d}")
print(f"{'Cross-element':20s} {ed['baseline']['cross']:>10d} {ed['bim_corrected']['cross']:>10d}")
print(f"{'Intra-element':20s} {ed['baseline']['intra']:>10d} {ed['bim_corrected']['intra']:>10d}")
print(f"\nCross-element reduction: {ed['cross_reduction_pct']:.1f}%")

## Per-Element Analysis

In [ ]:
print('Per-element breakdown:')
print('=' * 70)
print(f"{'Element':10s} {'n':>5s} {'Base Acc':>9s} {'BPC Acc':>9s} "
      f"{'Base CE':>8s} {'BPC CE':>7s} {'Base IE':>8s} {'BPC IE':>7s}")
print('-' * 70)
for elem, d in decomp['element_analysis'].items():
    print(f"{elem:10s} {d['n']:>5d} {d['baseline_acc']:>9.4f} {d['corrected_acc']:>9.4f} "
          f"{d['baseline_cross_err']:>8d} {d['corrected_cross_err']:>7d} "
          f"{d['baseline_intra_err']:>8d} {d['corrected_intra_err']:>7d}")
print('-' * 70)
print('CE = cross-element errors, IE = intra-element errors')

## Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: stacked bar (baseline vs BPC)
labels = ['Baseline', 'BIM-BPC']
cross = [ed['baseline']['cross'], ed['bim_corrected']['cross']]
intra = [ed['baseline']['intra'], ed['bim_corrected']['intra']]

ax1.bar(labels, intra, label='Intra-element', color='#FFC107')
ax1.bar(labels, cross, bottom=intra, label='Cross-element', color='#F44336')
ax1.set_ylabel('Error count')
ax1.set_title('Error decomposition')
ax1.legend()

# Right: per-element cross-error reduction
elements = list(decomp['element_analysis'].keys())
base_ce = [decomp['element_analysis'][e]['baseline_cross_err'] for e in elements]
bpc_ce = [decomp['element_analysis'][e]['corrected_cross_err'] for e in elements]

x = np.arange(len(elements))
w = 0.35
ax2.bar(x - w/2, base_ce, w, label='Baseline', color='#F44336', alpha=0.7)
ax2.bar(x + w/2, bpc_ce, w, label='BIM-BPC', color='#4CAF50', alpha=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(elements)
ax2.set_ylabel('Cross-element errors')
ax2.set_title('Cross-element errors by element type')
ax2.legend()

plt.tight_layout()
plt.savefig('../figures/nb03_error_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Findings

- **100% cross-element error elimination** across all element types.
- Intra-element errors remain unchanged (376 → 377; +1 due to EN 206 soft prior edge case).
- Deck has the highest baseline cross-element error rate (14/2,045 = 0.68%).
- Formal definition: ŷ is a cross-element error if ŷ ≠ y AND elem(ŷ) ≠ elem(y).